# DS Team 2: Exploratory Data Analysis (EDA)

**Purpose:** Explore the ShopFlow datasets to understand customer-product interactions, assess data quality, and identify patterns that will guide our Recommendation Engine design.

**Contract Reference:** `GET /recommend/{customer_id}?n=5` — returns top-N products with `predicted_score` (0.0–1.0), sorted descending.

In [ ]:
import sys
import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Add src to path to import data_loader
sys.path.append(os.path.abspath('../src'))
from data_loader import get_s3_data

# Set visual style
sns.set_palette('viridis')
plt.style.use('ggplot')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12

---
## 1. Load Datasets

In [ ]:
customers = get_s3_data('customers')
products = get_s3_data('products')
orders = get_s3_data('orders')
events = get_s3_data('events')

print(f'Customers: {customers.shape}')
print(f'Products:  {products.shape}')
print(f'Orders:    {orders.shape}')
print(f'Events:    {events.shape}')

---
## 2. Data Type Inspection

In [ ]:
print('=== CUSTOMERS ===')
print(customers.dtypes)
print()
customers.head(3)

In [ ]:
print('=== PRODUCTS ===')
print(products.dtypes)
print()
products.head(3)

In [ ]:
print('=== ORDERS ===')
print(orders.dtypes)
print()
orders.head(3)

In [ ]:
print('=== EVENTS ===')
print(events.dtypes)
print()
events.head(3)

---
## 3. Data Quality: Missing Values & Duplicates

In [ ]:
print('=== MISSING VALUES ===')
for name, df in [('Customers', customers), ('Products', products), ('Orders', orders), ('Events', events)]:
    missing = df.isnull().sum()
    missing_pct = (df.isnull().sum() / len(df) * 100).round(2)
    summary = pd.DataFrame({'missing_count': missing, 'missing_pct': missing_pct})
    summary = summary[summary['missing_count'] > 0]
    if len(summary) > 0:
        print(f'\n{name}:')
        print(summary)
    else:
        print(f'{name}: No missing values')

In [ ]:
print('=== DUPLICATE CHECK ===')
print(f'Duplicate customer IDs: {customers["customer_id"].duplicated().sum()}')
print(f'Duplicate product IDs:  {products["product_id"].duplicated().sum()}')
print(f'Duplicate order IDs:    {orders["order_id"].duplicated().sum()}')
print(f'Duplicate event IDs:    {events["event_id"].duplicated().sum()}')

---
## 4. Descriptive Statistics

In [ ]:
print('=== ORDERS: Numerical Summary ===')
orders.describe()

In [ ]:
print('=== PRODUCTS: Numerical Summary ===')
products.describe()

---
## 5. Customer Demographics Analysis

In [ ]:
# Loyalty Tier Distribution
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Loyalty tiers
loyalty_counts = customers['loyalty_tier'].value_counts()
axes[0].bar(loyalty_counts.index, loyalty_counts.values, color=sns.color_palette('viridis', len(loyalty_counts)))
axes[0].set_title('Customer Loyalty Tier Distribution')
axes[0].set_xlabel('Loyalty Tier')
axes[0].set_ylabel('Count')
for i, v in enumerate(loyalty_counts.values):
    axes[0].text(i, v + 200, str(v), ha='center', fontweight='bold')

# Signup date trend
customers['signup_date'] = pd.to_datetime(customers['signup_date'])
customers['signup_month'] = customers['signup_date'].dt.to_period('M')
signup_trend = customers['signup_month'].value_counts().sort_index()
axes[1].plot(signup_trend.index.astype(str), signup_trend.values, marker='o', linewidth=2)
axes[1].set_title('Customer Signup Trend Over Time')
axes[1].set_xlabel('Month')
axes[1].set_ylabel('New Signups')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

print(f'Total customers: {len(customers)}')
print(f'Loyalty tier breakdown:\n{loyalty_counts}')

---
## 6. Product Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Category distribution
cat_counts = products['category'].value_counts()
axes[0].pie(cat_counts.values, labels=cat_counts.index, autopct='%1.1f%%', startangle=90)
axes[0].set_title('Product Category Distribution')

# Price distribution
axes[1].hist(products['price'], bins=30, edgecolor='black', alpha=0.7)
axes[1].axvline(products['price'].mean(), color='red', linestyle='--', label=f"Mean: {products['price'].mean():.2f}")
axes[1].axvline(products['price'].median(), color='orange', linestyle='--', label=f"Median: {products['price'].median():.2f}")
axes[1].set_title('Product Price Distribution')
axes[1].set_xlabel('Price')
axes[1].set_ylabel('Count')
axes[1].legend()

plt.tight_layout()
plt.show()

print(f'Total products: {len(products)}')
print(f'Categories: {products["category"].nunique()}')
print(f'Price range: {products["price"].min():.2f} - {products["price"].max():.2f}')

---
## 7. Product Popularity (Top Purchased)

In [ ]:
# Top 10 most purchased products
top_products = orders['product_id'].value_counts().head(10)

# Merge with product names
top_df = top_products.reset_index()
top_df.columns = ['product_id', 'purchase_count']
top_df = top_df.merge(products[['product_id', 'product_name', 'category']], on='product_id', how='left')

plt.figure(figsize=(14, 6))
bars = plt.barh(range(len(top_df)), top_df['purchase_count'], color=sns.color_palette('viridis', 10))
plt.yticks(range(len(top_df)), [f"{row['product_name']} ({row['category']})" for _, row in top_df.iterrows()])
plt.xlabel('Purchase Count')
plt.title('Top 10 Most Purchased Products')
plt.gca().invert_yaxis()

for i, v in enumerate(top_df['purchase_count']):
    plt.text(v + 5, i, str(v), va='center', fontweight='bold')

plt.tight_layout()
plt.show()

---
## 8. Event Type Breakdown (Critical for Weighting)

This analysis determines how we weight implicit feedback signals: Views, Carts, and Purchases.

In [ ]:
event_counts = events['event_type'].value_counts()

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Bar chart
colors = {'view': '#2ecc71', 'cart': '#f39c12', 'purchase': '#e74c3c'}
bar_colors = [colors.get(e, '#95a5a6') for e in event_counts.index]
axes[0].bar(event_counts.index, event_counts.values, color=bar_colors)
axes[0].set_title('Event Type Distribution')
axes[0].set_ylabel('Count')
for i, v in enumerate(event_counts.values):
    axes[0].text(i, v + 1000, f'{v:,}', ha='center', fontweight='bold')

# Pie chart
axes[1].pie(event_counts.values, labels=event_counts.index, autopct='%1.1f%%', colors=bar_colors, startangle=90)
axes[1].set_title('Event Type Proportions')

plt.tight_layout()
plt.show()

print('Event Type Counts:')
print(event_counts)
print(f'\nEvent Type Percentages:')
print((event_counts / event_counts.sum() * 100).round(2))

---
## 9. Purchase Frequency per User & Cold Start Detection

In [ ]:
# How many orders does each customer have?
orders_per_user = orders.groupby('customer_id')['order_id'].nunique().reset_index()
orders_per_user.columns = ['customer_id', 'order_count']

# Unique products per customer
products_per_user = orders.groupby('customer_id')['product_id'].nunique().reset_index()
products_per_user.columns = ['customer_id', 'unique_products']

user_stats = orders_per_user.merge(products_per_user, on='customer_id')

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

axes[0].hist(user_stats['order_count'], bins=30, edgecolor='black', alpha=0.7)
axes[0].set_title('Distribution of Orders per Customer')
axes[0].set_xlabel('Number of Orders')
axes[0].set_ylabel('Number of Customers')
axes[0].axvline(user_stats['order_count'].mean(), color='red', linestyle='--', label=f"Mean: {user_stats['order_count'].mean():.1f}")
axes[0].legend()

axes[1].hist(user_stats['unique_products'], bins=30, edgecolor='black', alpha=0.7, color='orange')
axes[1].set_title('Unique Products per Customer')
axes[1].set_xlabel('Number of Unique Products')
axes[1].set_ylabel('Number of Customers')
axes[1].axvline(user_stats['unique_products'].mean(), color='red', linestyle='--', label=f"Mean: {user_stats['unique_products'].mean():.1f}")
axes[1].legend()

plt.tight_layout()
plt.show()

# Cold start analysis
all_customer_ids = set(customers['customer_id'].unique())
customers_with_orders = set(orders['customer_id'].unique())
customers_with_events = set(events['customer_id'].unique())
cold_start_customers = all_customer_ids - customers_with_orders - customers_with_events
order_only = customers_with_orders - customers_with_events
event_only = customers_with_events - customers_with_orders
both = customers_with_orders & customers_with_events

print('=== COLD START ANALYSIS ===')
print(f'Total customers:              {len(all_customer_ids)}')
print(f'Customers with orders:        {len(customers_with_orders)}')
print(f'Customers with events:        {len(customers_with_events)}')
print(f'Customers with BOTH:          {len(both)}')
print(f'Cold start (no history):      {len(cold_start_customers)} ({len(cold_start_customers)/len(all_customer_ids)*100:.1f}%)')

---
## 10. Category Interaction Analysis

In [ ]:
# Orders by category
orders_with_cat = orders.merge(products[['product_id', 'category']], on='product_id', how='left')

cat_popularity = orders_with_cat.groupby('category').agg(
    total_orders=('order_id', 'count'),
    unique_customers=('customer_id', 'nunique'),
    total_revenue=('amount', 'sum'),
    avg_order_value=('amount', 'mean')
).sort_values('total_orders', ascending=False).reset_index()

print('=== CATEGORY PERFORMANCE ===')
print(cat_popularity.to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

axes[0].barh(cat_popularity['category'], cat_popularity['total_orders'], color=sns.color_palette('viridis', len(cat_popularity)))
axes[0].set_title('Orders by Category')
axes[0].set_xlabel('Total Orders')
axes[0].invert_yaxis()

axes[1].barh(cat_popularity['category'], cat_popularity['total_revenue'], color=sns.color_palette('magma', len(cat_popularity)))
axes[1].set_title('Revenue by Category')
axes[1].set_xlabel('Total Revenue')
axes[1].invert_yaxis()

plt.tight_layout()
plt.show()

---
## 11. Temporal Patterns

In [ ]:
orders['timestamp'] = pd.to_datetime(orders['timestamp'])
events['timestamp'] = pd.to_datetime(events['timestamp'])

fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Orders over time (monthly)
orders['order_month'] = orders['timestamp'].dt.to_period('M')
monthly_orders = orders.groupby('order_month').size()
axes[0, 0].plot(monthly_orders.index.astype(str), monthly_orders.values, marker='o', linewidth=2)
axes[0, 0].set_title('Monthly Order Volume')
axes[0, 0].set_xlabel('Month')
axes[0, 0].set_ylabel('Orders')
axes[0, 0].tick_params(axis='x', rotation=45)

# Orders by day of week
orders['day_of_week'] = orders['timestamp'].dt.day_name()
day_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
dow_counts = orders['day_of_week'].value_counts().reindex(day_order)
axes[0, 1].bar(dow_counts.index, dow_counts.values, color=sns.color_palette('viridis', 7))
axes[0, 1].set_title('Orders by Day of Week')
axes[0, 1].set_ylabel('Orders')
axes[0, 1].tick_params(axis='x', rotation=45)

# Events over time (monthly)
events['event_month'] = events['timestamp'].dt.to_period('M')
monthly_events = events.groupby('event_month').size()
axes[1, 0].plot(monthly_events.index.astype(str), monthly_events.values, marker='s', linewidth=2, color='orange')
axes[1, 0].set_title('Monthly Event Volume')
axes[1, 0].set_xlabel('Month')
axes[1, 0].set_ylabel('Events')
axes[1, 0].tick_params(axis='x', rotation=45)

# Device distribution in events
device_counts = events['device'].value_counts()
axes[1, 1].pie(device_counts.values, labels=device_counts.index, autopct='%1.1f%%', startangle=90)
axes[1, 1].set_title('Event Device Distribution')

plt.tight_layout()
plt.show()

---
## 12. Interaction Matrix Sparsity Analysis

This is the most critical metric for the Recommendation Engine. High sparsity means we need ALS (handles sparse data well) rather than basic similarity.

In [ ]:
# From orders only
n_users_orders = orders['customer_id'].nunique()
n_items_orders = orders['product_id'].nunique()
n_interactions_orders = len(orders)
sparsity_orders = 1 - (n_interactions_orders / (n_users_orders * n_items_orders))

# From events only
n_users_events = events['customer_id'].nunique()
n_items_events = events['product_id'].nunique()
n_interactions_events = len(events)
sparsity_events = 1 - (n_interactions_events / (n_users_events * n_items_events))

# Combined (unique user-product pairs across both)
order_pairs = orders[['customer_id', 'product_id']].drop_duplicates()
event_pairs = events[['customer_id', 'product_id']].drop_duplicates()
all_pairs = pd.concat([order_pairs, event_pairs]).drop_duplicates()
n_users_all = all_pairs['customer_id'].nunique()
n_items_all = all_pairs['product_id'].nunique()
sparsity_combined = 1 - (len(all_pairs) / (n_users_all * n_items_all))

print('=== INTERACTION MATRIX SPARSITY ===')
print(f'Orders Only:   {n_users_orders} users x {n_items_orders} products = {n_interactions_orders} interactions | Sparsity: {sparsity_orders:.4%}')
print(f'Events Only:   {n_users_events} users x {n_items_events} products = {n_interactions_events} interactions | Sparsity: {sparsity_events:.4%}')
print(f'Combined:      {n_users_all} users x {n_items_all} products = {len(all_pairs)} unique pairs | Sparsity: {sparsity_combined:.4%}')
print()
print(f'Average interactions per user (orders): {n_interactions_orders / n_users_orders:.1f}')
print(f'Average interactions per user (events): {n_interactions_events / n_users_events:.1f}')
print(f'Average unique products per user (combined): {len(all_pairs) / n_users_all:.1f}')

---
## 13. Payment Method Analysis

In [ ]:
payment_counts = orders['payment_method'].value_counts()

plt.figure(figsize=(10, 6))
plt.bar(payment_counts.index, payment_counts.values, color=sns.color_palette('viridis', len(payment_counts)))
plt.title('Payment Method Distribution')
plt.ylabel('Count')
for i, v in enumerate(payment_counts.values):
    plt.text(i, v + 500, f'{v:,}', ha='center', fontweight='bold')
plt.tight_layout()
plt.show()

---
## 14. Summary & Key Findings

In [ ]:
print('=' * 60)
print('EDA SUMMARY — DS Team 2: Recommendation Engine')
print('=' * 60)
print()
print('DATA OVERVIEW')
print(f'  Customers: {len(customers):,}')
print(f'  Products:  {len(products):,}')
print(f'  Orders:    {len(orders):,}')
print(f'  Events:    {len(events):,}')
print()
print('INTERACTION MATRIX')
print(f'  Combined sparsity:            {sparsity_combined:.4%}')
print(f'  Unique user-product pairs:    {len(all_pairs):,}')
print(f'  Avg products per user:        {len(all_pairs) / n_users_all:.1f}')
print()
print('COLD START')
print(f'  Users with no history:        {len(cold_start_customers):,} ({len(cold_start_customers)/len(all_customer_ids)*100:.1f}%)')
print()
print('KEY DECISIONS FOR MODELING')
print('  1. High sparsity -> ALS (Matrix Factorization) is preferred over Cosine Similarity')
print('  2. Event weighting: view=1, cart=2, purchase=5 (based on signal strength)')
print('  3. Cold start users will receive popularity-based fallback recommendations')
print('  4. Scores will be normalized to [0.0, 1.0] per contract requirement')
print()
print('NEXT STEP: Feature Engineering (preprocessing.py)')